# Create Templeton Prize Awards (PRIZE PATTERN, same-funder-second-priority)

Ingest of the **Templeton Prize**, awarded annually since 1973 by the **John Templeton Foundation** to "a living person who has made an exceptional contribution to affirming life's spiritual dimension." Laureates have included Mother Teresa (1973), the Dalai Lama (2012), Desmond Tutu (2013), Francis Collins (2020), Marcelo Gleiser (2019), and Simon Conway Morris (2026). Source: `templetonprize.org/templeton-prize-winners-2/` — the foundation's own site (WordPress, no auth, no aggregator). Method #5 on the runbook ladder (static-HTML scrape).

**Awarding body:** John Templeton Foundation — `F4320306193` (US, DOI `10.13039/100000925`).

**This is a same-funder, second-priority ingest** — a new precedent worth documenting:

The John Templeton Foundation is **already in `CreateAwards.ipynb` at priority 39** for the ~5,956 Templeton GRANTS (its main research-funding output). This notebook adds a SECOND priority slot (priority 93, `provenance='templeton_prize'`) for the Templeton PRIZE — operationally a separate program: flagship $1.4M annual honor vs. a sprawling research-grant portfolio. The two coexist cleanly because:
- The Step 3 `DELETE` uses `(provenance, priority)` as the key — `'templeton_prize'`+`93` won't touch any rows owned by the existing Templeton grant ingest.
- Per-row `id = abs(xxhash64(funder_id ':' funder_award_id)) % 9e9`. A funder_award_id like `templeton-prize-1983-aleksandr-solzhenitsyn` hashes to a distinct ID from any grant award_id under the same funder.
- The `openalex_awards` dedup is `ROW_NUMBER() OVER (PARTITION BY id ORDER BY priority ASC)`. Distinct IDs mean no cross-priority dedup happens; both rows coexist as separate awards.

This precedent generalizes: a single `funder_id` may legitimately appear at multiple priorities when the funder runs operationally separate award programs (a flagship prize + a research-grant program). Future contributors should not hesitate to add a same-funder second-priority entry when a flagship prize is operationally distinct from the funder's main grant output (e.g., HHMI's Investigator Program vs. its Medical Institute fellowships; NIH's various Director's Awards vs. its broader grant portfolio).

**Corpus** (verified 2026-05-21 full local scrape): **56 laureate rows** spanning **1973-2026** (54 years × 1 laureate/year with some joint-year exceptions). Distribution: 7 in 1970s, 11 in 1980s, 11 in 1990s, 10 in 2000s, 10 in 2010s, 7 in 2020s.

**Schema:**
- `funder_award_id` = `templeton-prize-{year}-{slug}` (e.g., `templeton-prize-1983-aleksandr-solzhenitsyn`). Verified unique across the 56-row corpus.
- `lead_investigator` is PERSON-LEVEL: `given_name` + `family_name` parsed via the script's `split_name()` helper (runbook §2.4.1) with extended honorific list covering religious titles common among Templeton laureates (Mother, Brother, Father, Lord, Dame, Sir, Reverend, Rabbi, Archbishop, Bishop, His Holiness, His All Holiness Ecumenical Patriarch). 52/56 rows have `given_name`; 4 mononymic laureates (Mother Teresa 1973, Brother Roger 1974, Lord MacLeod 1989, Lord Jakobovits 1991) carry only `family_name` after the title is stripped — that's accurate per cultural / monastic naming convention, not a data bug.
- `affiliation` is NULL — the Templeton Prize honors individuals for spiritual contributions; the foundation does not publish per-laureate institutional affiliation on the listing page. Bio enrichment from the `/laureate/{slug}/` detail pages could populate this in a follow-up; documented as a Step 0 follow-up in the tracker.
- `funder_scheme = 'Templeton Prize'`. Single program.
- `funding_type = 'prize'`.
- `declined` always FALSE.

**Amount choice — constant GBP 1,100,000 per laureate:**
The foundation's `templeton-prize-history/quick-facts/` page states verbatim: **"The Templeton Prize is a monetary award in the amount of £1,100,000 sterling."** Verified 2026-05-21. The amount was set in 2001 to exceed the Nobel Prize and is reviewed periodically; the foundation publishes only the current rule. We ship GBP 1,100,000 uniformly across all years and document the assumption. `currency='GBP'`. **§6.7 amount-coverage NOT waived** — 100% expected.

**Source authority** — `templetonprize.org` is the John Templeton Foundation's own site. WordPress, server-side rendered listing — no FacetWP / no admin-ajax / no Playwright needed. Method #5 (static-HTML scrape) on the runbook ladder.

**Prerequisites**: Run `scripts/local/templeton_prize_to_s3.py` first to scrape the single listing page (~10 sec wall-clock), parse, write parquet, upload to S3.

**Data source**: `https://www.templetonprize.org/templeton-prize-winners-2/`

**S3 location**: `s3a://openalex-ingest/awards/templeton_prize/templeton_prize_laureates.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.templeton_prize_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/templeton_prize/templeton_prize_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.templeton_prize_raw;

## Step 1.5: Inspect raw + money/currency scan + coverage acknowledgment

All source columns from the listing-page scrape. **Verified per-row coverage on the full extraction** (56 laureate rows, validated 2026-05-21):
- 100% `funder_award_id`, `name`, `year`, `slug`, `occupation`, `landing_page_url`
- 100% `amount` = constant £1,100,000 GBP per laureate (foundation's documented rule)
- 93% `given_name` (52/56 — 4 mononymic religious-title laureates carry only family_name)
- 100% `family_name`
- 100% `description` (composed from `occupation`)
- 54 distinct years 1973-2026
- `affiliation_name` is intentionally NULL — listing page doesn't expose it; would require per-laureate detail-page enrichment (Step 0 follow-up)

Source columns: `funder_award_id`, `slug`, `name`, `given_name`, `family_name`, `year`, `year_end`, `occupation`, `display_name`, `description`, `amount`, `currency`, `start_date`, `end_date`, `landing_page_url`, `image_url`, `decade_class`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.templeton_prize_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.templeton_prize_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — amount is the foundation's documented constant
-- £1,100,000 GBP ceiling. Verify the column shipped correctly.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.templeton_prize_raw;

In [ ]:
%sql
-- Year coverage sanity — should span 1973-2026 with ~1 laureate/year average.
SELECT MIN(TRY_CAST(year AS INT)) AS min_yr,
       MAX(TRY_CAST(year AS INT)) AS max_yr,
       COUNT(DISTINCT year) AS distinct_years,
       COUNT(*) AS total_laureates
FROM openalex.awards.templeton_prize_raw;

## Step 1.6: Fail-fast — verify John Templeton Foundation funder row exists

**Note:** This is the SAME funder_id already used at priority 39 for Templeton grants. Both ingests coexist via distinct `provenance` keys.

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306193;

In [ ]:
%sql
-- Sanity: confirm the existing Templeton GRANT rows at priority 39 exist
-- and won't be touched by our Step 3 DELETE (which targets priority 93 +
-- provenance='templeton_prize' only).
SELECT provenance, priority, COUNT(*) AS existing_rows
FROM openalex.awards.openalex_awards_raw
WHERE funder_id = 4320306193
GROUP BY provenance, priority
ORDER BY priority;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.templeton_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306193  -- John Templeton Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    'Templeton Prize' AS funder_scheme,
    'templeton_prize' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator is PERSON-LEVEL: given_name + family_name parsed
    -- in the script. affiliation is NULL because the listing page doesn't
    -- expose per-laureate affiliations (Step 0 follow-up could enrich
    -- from /laureate/{slug}/ detail pages). For the 4 mononymic laureates
    -- (Mother Teresa, Brother Roger, Lord MacLeod, Lord Jakobovits), the
    -- given_name is NULL by design — that matches cultural / monastic
    -- naming convention.
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,  -- /laureate/{slug}/ detail page
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.templeton_prize_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 93

Priority 93 chosen — distinct from the existing Templeton GRANT entry at priority 39. The DELETE clause keys on `(provenance, priority)`, so this won't touch the existing grant rows.

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'templeton_prize' AND priority = 93;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    93 AS priority  -- Templeton Prize priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.templeton_prize_awards;

## Step 6: Verification

Full §6.1-6.7. Amount-coverage NOT waived (constant £1.1M per laureate). Verified expectations from the 2026-05-21 extraction:
- Row count: **56** (full roster across 1973-2026)
- 100% `pct_amount` (constant £1,100,000 GBP per laureate)
- `currencies = [GBP]`
- 1 distinct `funder_scheme` ('Templeton Prize'), 1 distinct `funding_type` ('prize')
- 54 distinct years (1973-2026)
- 100% `lead_investigator` populated (52/56 with `given_name`; 4 mononymic religious-title laureates legitimately carry only `family_name`)
- `declined = FALSE` everywhere
- **Cross-check**: confirm priority 39 Templeton GRANT rows are untouched by this ingest (Step 3 DELETE targets priority 93 + provenance='templeton_prize' only)

In [ ]:
%sql
SELECT COUNT(*) AS total_templeton_prize_award_rows FROM openalex.awards.templeton_prize_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.templeton_prize_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_dates,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.templeton_prize_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.templeton_prize_awards;

In [ ]:
%sql
-- Year distribution
SELECT start_year,
       COUNT(*) AS rows,
       ROUND(SUM(amount)/1e6, 2) AS total_gbp_millions
FROM openalex.awards.templeton_prize_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 100;

In [ ]:
%sql
-- Sample notable laureates (recent + earliest)
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(description, 1, 60) AS occupation,
       amount, start_year
FROM openalex.awards.templeton_prize_awards
WHERE start_year >= 2024 OR start_year <= 1976
ORDER BY start_year DESC
LIMIT 12;

In [ ]:
%sql
-- Mononymic laureate spot-check (Mother Teresa, Brother Roger, Lord MacLeod,
-- Lord Jakobovits should appear here with given_name IS NULL).
SELECT lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       start_year,
       SUBSTRING(display_name, 1, 80) AS title
FROM openalex.awards.templeton_prize_awards
WHERE lead_investigator.given_name IS NULL
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.templeton_prize_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Same-funder cross-priority sanity: after Step 3 INSERT, F4320306193 should
-- have rows at BOTH priority 39 (grants) AND priority 93 (this prize). The
-- two coexist because their IDs (xxhash of funder_id+funder_award_id) are
-- distinct and the openalex_awards dedup PARTITIONs BY id.
SELECT provenance, priority, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE funder_id = 4320306193
GROUP BY provenance, priority
ORDER BY priority;

In [ ]:
%sql
-- Declined boolean must be FALSE everywhere.
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.templeton_prize_awards
GROUP BY declined;